# Problem Set 1 — Building a Strategic Portfolio

Complete this notebook and submit it as **`submission.ipynb`**.

Rules:
- Run **Kernel → Restart & Run All** before submitting.
- Keep every function and variable name exactly as written.
- Write code only where you see `# TODO`.

## Autograder contract

The names below must not be renamed.

### Functions
- `test_normality(returns)`
- `portfolio_return(weights)`
- `portfolio_volatility(weights)`
- `portfolio_sharpe(weights)`
- `calculate_efficient_frontier(mu, Sigma)`
- `find_tangency_portfolio(mu, Sigma, risk_free_rate)`
- `optimal_complete_portfolio(tangency_weights, mu, Sigma, rf=0.02, A=9)`
- `perform_risk_analysis(portfolio_weights, returns)`
- `plot_tangency_weights(tickers, weights)`

### Module-level variables
`tickers`, `start_date`, `end_date`, `asset_prices`, `market_prices`,
`returns`, `returns_market`, `risk_free_rate`,
`mu`, `Sigma`,
`tangency_weights`, `y_star`,
`normality_results`, `risk_table`,
`expected_perf`, `expected_risk`, `realized_perf`, `realized_risk`,
`portfolio_fig`

In [2]:
from hashlib import new

import scipy
import scipy as sp
from scipy.ndimage import variance


In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.optimize import minimize
from scipy.stats import skew, kurtosis, chi2
import scipy.stats

## 1. Load data and compute returns

Download price data for at least five assets.

Define:
- `tickers` — list of ticker symbols
- `start_date`, `end_date` — strings
- `asset_prices` — adjusted close prices (DataFrame)
- `returns` — daily returns from `asset_prices`
- `market_prices` — benchmark prices (use `"SPY"`, single-column DataFrame)
- `returns_market` — benchmark return series
- `risk_free_rate` — annual risk-free rate (use `0.02`)

In [4]:
tickers        = ["SPY", "QQQ", "GLD", "TLT", "VNQ", "^GDAXI"]
start_date     = "2010-01-01"
end_date       = "2024-12-31"
risk_free_rate = 0.02

# done TODO: download price data using yf.download()
# done TODO: store closing prices in asset_prices (DataFrame)
# done TODO: compute daily returns from asset_prices and store in returns
# done TODO: define market_prices (single-column, SPY) and returns_market

asset_prices = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False
)["Adj Close"]
returns        = asset_prices.pct_change().dropna()
market_prices  = asset_prices[["SPY"]]
returns_market = market_prices.pct_change().dropna()

[*********************100%***********************]  6 of 6 completed


## Task 1 — Jarque-Bera Normality Test  [2 pts]

Implement `test_normality(returns)`.

Jarque-Bera formula:
$$JB = \frac{n}{6}\left(S^2 + \frac{(K-3)^2}{4}\right)$$

- $S$ = skewness (`stats.skew`)
- $K$ = Pearson kurtosis (`stats.kurtosis(data, fisher=False)`)
- p-value = `1 - chi2.cdf(JB, df=2)`

Return a DataFrame with one row per asset, columns:
`"JB Statistic"`, `"p-value"`, `"Skewness"`, `"Kurtosis"`

In [6]:
def test_normality(returns):
    """Jarque-Bera test for each asset return series."""
    results = {}
    for asset in returns.columns:
        data = returns[asset].dropna()
        n    = len(data)

        # done TODO: compute skewness
        skew_val = scipy.stats.skew(data)
        #done TODO: compute Pearson kurtosis
        kurt_val = scipy.stats.kurtosis(data, fisher=False)

        # done TODO: compute JB statistic (see formula in task header)
        jb_stat = (n / 6) * (skew_val**2 + 0.25 * (kurt_val - 3)**2)
        # done TODO: compute p-value from the JB statistic
        p_value = 1 - scipy.stats.chi2.cdf(jb_stat, df=2)

        results[asset] = {
            "JB Statistic": jb_stat,
            "p-value":      p_value,
            "Skewness":     skew_val,
            "Kurtosis":     kurt_val,
        }
    return pd.DataFrame(results).T


normality_results = test_normality(returns) if returns is not None else None
normality_results

,JB Statistic,p-value,Skewness,Kurtosis
GLD,3273.422033,0.0,-0.409988,7.631585
QQQ,5582.494211,0.0,-0.385758,9.093845
SPY,18150.713430,0.0,-0.553437,14.020405
TLT,2041.860735,0.0,0.085821,6.710900
VNQ,40192.877559,0.0,-0.785541,19.406757
^GDAXI,8198.999435,0.0,-0.262111,10.425593


## Task 2 — Opportunity Set & Efficient Frontier  [2 pts]

1. Compute `mu` and `Sigma` from `returns`.
2. Implement the three helper functions.
3. Implement `calculate_efficient_frontier(mu, Sigma)`.

Formulas:
- $\mu_p = w^\top \mu$
- $\sigma_p = \sqrt{w^\top \Sigma w}$
- $\text{Sharpe} = (\mu_p - r_f) / \sigma_p$

`calculate_efficient_frontier` must return a list of weight arrays (≥ 5 portfolios).
For each target return, solve: minimise $\sigma_p$ subject to $\sum w_i = 1$,
$w_i \geq 0$, and $\mu_p = \text{target}$.

In [7]:
# done TODO: compute annualised expected returns
mu    = returns.mean() * 252 # 252 is the number of trading days in the year. Some sources use 251 or 256 as well.

# done TODO: compute annualised covariance matrix
Sigma = returns.cov() * 252


def portfolio_return(weights):
    """Expected portfolio return: w^T mu."""
    weights_array = np.asarray(weights)
    return np.dot(weights_array, mu)

def portfolio_volatility(weights):
    """Portfolio volatility: sqrt(w^T Sigma w)."""
    # done TODO: return the square root of the quadratic form w^T Sigma w
    weights_array = np.asarray(weights)
    variance = weights_array.T @ Sigma @ weights_array
    return float(np.sqrt(variance))

def portfolio_sharpe(weights):
    """Sharpe ratio: (return - rf) / volatility."""
    # done TODO: use portfolio_return and portfolio_volatility
    volatility = portfolio_volatility(weights)
    if volatility == 0: # we cannot divide with 0, in order to accept the zero volatility situation I defined it as negative infinity
        return -np.inf

    return float(portfolio_return(weights) - risk_free_rate) / volatility



def calculate_efficient_frontier(mu, Sigma): #??????????????
    """Return a list of long-only weight vectors on the efficient frontier."""
    # TODO: for a grid of target returns, minimise portfolio_volatility
    #       subject to: weights sum to 1, weights >= 0, return == target
    # Return a list of weight arrays (at least 5 portfolios)



## Task 3 — Tangency Portfolio & Optimal Complete Portfolio  [2 pts]

1. Implement `find_tangency_portfolio(mu, Sigma, risk_free_rate)`.
   Maximise the Sharpe ratio (minimise its negative) subject to long-only weights summing to 1.
2. Implement `optimal_complete_portfolio(tangency_weights, mu, Sigma, rf=0.02, A=9)`.
   Return `y_star` (the share invested in the tangency portfolio):
   $$y^* = \min\!\left(\frac{\mu_p - r_f}{A\,\sigma_p^2},\;1\right)$$
3. Assign `tangency_weights` and `y_star` at module level.

In [25]:
def find_tangency_portfolio(mu, Sigma, risk_free_rate):
    """Long-only portfolio that maximises the Sharpe ratio."""
    # done TODO: minimise -portfolio_sharpe(w)
    #       subject to: weights sum to 1, weights >= 0

    mu_array = np.asarray(mu)
    S = np.asarray(Sigma)
    n = len(mu_array)

    def negative_sharpe_calculation(weights):
        portfolio_return = float(np.dot(weights, mu_array))
        portfolio_volatility = float(np.sqrt(weights.T @ S @ weights))

        if portfolio_volatility == 0: return np.inf

        return -((portfolio_return - risk_free_rate) / portfolio_volatility)


    start_weights = np.ones(n) / n # I assume it is equally distributed
    bounds = [(0, 1) for _ in range(n)]

    constraints = {
        "type": "eq",
        "fun": lambda weights: np.sum(weights) - 1
    }

    result = scipy.optimize.minimize(
        fun = negative_sharpe_calculation, #we want to minimize the negative sharpe so that we actually find maximum
        x0 = start_weights, #our start point is the equal distribution
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )
    return result.x


def optimal_complete_portfolio(tangency_weights, mu, Sigma, rf=0.02, A=9):
    """Return y_star = min((Rp - rf) / (A * sigma_p^2), 1)."""
    # TODO: compute expected return and volatility of the tangency portfolio
    # TODO: apply the formula from the task header and clip to [0, 1]
    weights = np.asarray(tangency_weights)
    mu = np.asarray(mu)
    S = np.asarray(Sigma)

    mu_p = float(np.dot(weights, mu))
    #variance_p = float(weights.T @ S @ weights)
    sigma_p = portfolio_volatility(weights)**2

    y_star_val = (mu_p - rf) / (A * sigma_p)
   # return min(y_star_val, 1)
    return float(np.clip(y_star_val, 0, 1))


# none TODO: call the functions above and assign the results
tangency_weights = find_tangency_portfolio(mu, Sigma, risk_free_rate)
y_star           = optimal_complete_portfolio(tangency_weights, mu, Sigma, risk_free_rate , A=9)

## Task 4 — OLS Risk Decomposition  [2 pts]

Implement `perform_risk_analysis(portfolio_weights, returns)`.

Regression model:
$$R_p - R_f = \alpha + \beta\,(R_m - R_f) + \varepsilon$$

Risk decomposition:
$$\sigma_p^2 = (\beta\,\sigma_m)^2 + \sigma_\varepsilon^2$$

Return a **dict** with these keys (exact spelling):
`"Beta (β)"`, `"Alpha (α)"`, `"R-squared"`,
`"Total Return"`, `"Systematic Return (β)"`, `"Idiosyncratic Return (α)"`,
`"Total Risk"`, `"Systematic Risk (βσₘ)"`, `"Idiosyncratic Risk (σₑ)"`

All return/risk values must be **annualised** (× 252 or × √252).

In [ ]:
def perform_risk_analysis(portfolio_weights, returns):
    """OLS decomposition of portfolio return and risk."""
    # TODO: construct the portfolio return series from weights and returns
    # TODO: compute excess returns for the portfolio and the market benchmark
    # TODO: run an OLS regression (use statsmodels)
    # TODO: extract beta, alpha, and residuals
    # TODO: annualise all return and risk values
    # TODO: return a dict with all required keys (see task header)
    raise NotImplementedError


risk_table = perform_risk_analysis(tangency_weights, returns) if tangency_weights is not None else None

## Task 5 — Expected vs. Realized Performance  [0.5 pts]

Split returns 70 % training / 30 % test.
Compute four scalar variables using `tangency_weights`:

- `expected_perf`  — $\sum_i w_i \mu_i$ from training data (annualised)
- `expected_risk`  — $\sqrt{w^\top \Sigma_{\text{train}} w}$ (annualised)
- `realized_perf`  — mean of daily portfolio returns on test data × 252
- `realized_risk`  — std of daily portfolio returns on test data × √252

In [ ]:
# TODO: split returns into 70 % training and 30 % test
# TODO: compute expected_perf and expected_risk from the training period
# TODO: compute realized_perf and realized_risk from the test period

expected_perf = None
expected_risk = None
realized_perf = None
realized_risk = None

## Task 6 — Visualize Portfolio Weights  [0.5 pts]

Implement `plot_tangency_weights(tickers, weights)`:

1. Create a **bar chart** showing each asset's weight.
2. Set the title to `"Tangency Portfolio Weights"`.
3. Label x-axis `"Assets"`, y-axis `"Weight"`.
4. **Return the `Figure` object** (required by the autograder).

Example skeleton:
```python
fig, ax = plt.subplots()
ax.bar(tickers, weights)
ax.set_title("Tangency Portfolio Weights")
ax.set_xlabel("Assets")
ax.set_ylabel("Weight")
return fig
```

In [ ]:
def plot_tangency_weights(tickers, weights):
    """Bar chart of tangency portfolio weights. Must return the Figure."""
    # TODO: create a bar chart with title and axis labels, then return the Figure
    raise NotImplementedError


portfolio_fig = (
    plot_tangency_weights(tickers, tangency_weights)
    if tangency_weights is not None else None
)

## Submission checklist

- [ ] Filename is exactly `submission.ipynb`.
- [ ] **Kernel → Restart & Run All** completes without errors.
- [ ] No function still raises `NotImplementedError`.
- [ ] No required variable is still `None`.
- [ ] `report.pdf` is ready.

Submit exactly two files: `submission.ipynb` and `report.pdf`.

## Academic integrity

Your work must be your own. Submissions with > 80 % similarity to another
student may receive zero points. Using AI tools to understand concepts is
allowed, but the final code and report must be yours.